# Open-weight LLMs, one sentence per prompt

Qwen2.5-7B-Instruct and Qwen2.5-14B-Instruct in 4-bit on the free T4, classifying each sentence in its own prompt (zero-shot and few-shot). Set MODEL_ID and restart the session between models.

In [ ]:
# ===== STEP 6c: open-weight LLM, ONE SENTENCE PER PROMPT (no batching, so no label-alignment errors) =====
# IMPORTANT: start from a fresh session (Runtime > Restart session) so no other model is in GPU memory.
# Runtime: T4 GPU. Set MODEL_ID below: run once for 7B, then restart the session and run again for 14B.
!pip -q install -U bitsandbytes accelerate

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"        # second run: "Qwen/Qwen2.5-14B-Instruct"
CONDITIONS = [("few_shot_position", True, True),     # the fairest single comparison; run this first
              ("zero_shot", False, False),
              ("zero_shot_position", True, False)]
# To save GPU time on the 14B run, keep only the first condition: CONDITIONS = CONDITIONS[:1]

import os, json, time, re, gc
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import f1_score, accuracy_score, classification_report
from google.colab import drive
drive.mount("/content/drive")

DATA = "/content/drive/MyDrive/sci-classify/data"; RES = "/content/drive/MyDrive/sci-classify/results"
LABELS = ["BACKGROUND", "OBJECTIVE", "METHODS", "RESULTS", "CONCLUSIONS"]
test_llm = pd.read_parquet(f"{DATA}/test_llm_1000.parquet")
train = pd.read_parquet(f"{DATA}/train.parquet")

# Never silently reuse a model left over from an earlier cell
for v in ("model", "tok", "MODEL"):
    globals().pop(v, None)
gc.collect(); torch.cuda.empty_cache()
print("GPU:", torch.cuda.get_device_name(0), f"| free memory before loading: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map={"": 0}, low_cpu_mem_usage=True,      # whole model on the GPU, or fail loudly
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4"))
model.eval()
MODEL = MODEL_ID.split("/")[-1] + " (4-bit)"
SLUG = MODEL_ID.split("/")[-1]
print("Loaded:", MODEL, f"| GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

DEFS = ("BACKGROUND: context, prior knowledge or the problem being addressed.\n"
        "OBJECTIVE: the aim, purpose or hypothesis of this study.\n"
        "METHODS: design, participants, interventions, measurements or analysis.\n"
        "RESULTS: findings, numbers and statistical outcomes.\n"
        "CONCLUSIONS: interpretation, implications or recommendations.\n")
EXAMPLES = train.groupby("label").sample(n=2, random_state=7)          # the same 10 examples as before

def build_prompt(r, use_position, few_shot):
    p = ("Classify this sentence from the abstract of a randomised controlled trial by its rhetorical role.\n"
         "Labels:\n" + DEFS)
    if use_position:
        p += "The sentence's position in its abstract is given, e.g. [3 of 9].\n"
    if few_shot:
        p += "\nExamples:\n" + "\n".join(
            (f"[{e.line_number + 1} of {e.total_lines}] " if use_position else "") + f"{e.text} -> {e.label}"
            for e in EXAMPLES.itertuples()) + "\n"
    p += "\nSentence: " + (f"[{r.line_number + 1} of {r.total_lines}] " if use_position else "") + r.text
    p += "\nAnswer with exactly one label: BACKGROUND, OBJECTIVE, METHODS, RESULTS or CONCLUSIONS."
    return p

def parse(text):
    found = [(text.upper().find(l), l) for l in LABELS if l in text.upper()]
    return min(found)[1] if found else "INVALID"                      # the first label the model wrote

@torch.no_grad()
def classify(prompt):
    enc = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)
    if not hasattr(enc, "keys"): enc = {"input_ids": enc}
    enc = {k: v.to(model.device) for k, v in enc.items()}
    out = model.generate(**enc, max_new_tokens=6, do_sample=False, pad_token_id=tok.eos_token_id)
    return parse(tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True))

def run(name, use_position, few_shot):
    path = f"{RES}/step6_single_{SLUG}_{name}.json"
    done = {int(k): v for k, v in json.load(open(path)).items()} if os.path.exists(path) else {}
    t0, n_new = time.time(), 0
    for i, r in enumerate(test_llm.itertuples()):
        if i in done: continue
        done[i] = classify(build_prompt(r, use_position, few_shot)); n_new += 1
        if n_new % 50 == 0:
            json.dump(done, open(path, "w")); print(f"{name}: {len(done)}/1000", end="\r")
    json.dump(done, open(path, "w"))
    preds = [done[i] for i in range(len(test_llm))]
    ms = (time.time() - t0) / max(n_new, 1) * 1000 if n_new >= 50 else None
    print(f"\n{SLUG} {name:20s} macro-F1 {f1_score(test_llm.label, preds, average='macro', labels=LABELS):.3f}  "
          f"accuracy {accuracy_score(test_llm.label, preds):.3f}  invalid {preds.count('INVALID')}"
          + (f"  {ms:.0f} ms/sentence" if ms else ""))
    return preds, ms

results, timing = {}, {}
for name, pos, fs in CONDITIONS:
    results[name], timing[name] = run(name, pos, fs)

best = max(results, key=lambda k: f1_score(test_llm.label, results[k], average="macro", labels=LABELS))
print(f"\nPer-class F1, best setting ({best}):")
print(classification_report(test_llm.label, results[best], labels=LABELS, digits=3, zero_division=0))
json.dump({"model": MODEL, "preds": results, "ms": timing}, open(f"{RES}/step6_single_{SLUG}_all.json", "w"))
print("Saved to", RES)
